In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import lines as mlines

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFECV, SelectKBest, f_regression, mutual_info_regression, RFE, GenericUnivariateSelect, SelectPercentile
from sklearn.model_selection import KFold, train_test_split
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

from itertools import product
from joblib import Parallel, delayed
from sklearn.base import clone
from tqdm.notebook import tqdm

### Function utils definitions

In [ ]:
######## ------  defining feature selection functions ------ ########

#### SelectKBest on the training set ####
def select_k_best_features(X_train, y_train, X_internal_test, y_internal_test, X_external_test, y_external_test, k, 
                           scoring_function=f_regression, 
                           regressor=KNeighborsRegressor(n_neighbors=10, weights='distance' , p=1), 
                           add_rxn_ohe=False,
                           plot=True):
    """
    Select the k best features based on the specified scoring function and regressor.
    Parameters:
    X_train (pd.DataFrame): Training feature set.
    y_train (pd.Series): Training target variable.
    X_internal_test (pd.DataFrame): Internal test feature set.
    y_internal_test (pd.Series): Internal test target variable.
    X_external_test (pd.DataFrame): External test feature set.
    y_external_test (pd.Series): External test target variable.
    k (int): Number of top features to select.
    scoring_function (function): Scoring function to evaluate the features (default: f_regression).
    regressor (sklearn regressor): Regressor to evaluate the selected features (default: KNeighborsRegressor).
    plot (bool): Whether to plot the predicted vs experimental values (default: True).
    Returns:
    tuple: A tuple containing the selected feature names, R2 scores, and MAE scores for the training, internal test, and external test sets.
    """

    # define selector and select the k best features on the training set
    selector = SelectKBest(score_func=scoring_function, k=k) 
    _ = selector.fit_transform(X_train, y_train)
    
    selected_feature_indices = selector.get_support(indices=True)
    selected_feature_names = X_train.columns[selected_feature_indices]

    if add_rxn_ohe and "rxn" not in selected_feature_names:
        selected_feature_names = selected_feature_names.tolist() + ["rxn"]

    # transform the internal and external test sets using the same selector
    X_train_selected = X_train[selected_feature_names]
    X_internal_test_selected = X_internal_test[selected_feature_names]
    X_external_test_selected = X_external_test[selected_feature_names]



    #print("Selected feature indices:", selected_feature_indices)
    if plot:
        print("Selected feature names:", selected_feature_names)

    # Fit the regressor on the training set with the selected features and evaluate on the training, internal test, and external test sets
    regressor.fit(X_train_selected, y_train)

    y_train_pred = regressor.predict(X_train_selected)
    y_internal_test_pred = regressor.predict(X_internal_test_selected)
    y_external_test_pred = regressor.predict(X_external_test_selected)

    r2_train = r2_score(y_train, y_train_pred)
    r2_internal_test = r2_score(y_internal_test, y_internal_test_pred)
    r2_external_test = r2_score(y_external_test, y_external_test_pred)

    mae_train = np.mean(np.abs(y_train - y_train_pred))
    mae_internal_test = np.mean(np.abs(y_internal_test - y_internal_test_pred))
    mae_external_test = np.mean(np.abs(y_external_test - y_external_test_pred))

    # plot 
    if plot:
        print (f"R2 train: {round(r2_train, 2)}, internal test: {round(r2_internal_test, 2)}, external test: {round(r2_external_test, 2)}")
        print (f"MAE train: {round(mae_train, 2)}, internal test: {round(mae_internal_test, 2)}, external test: {round(mae_external_test, 2)}")
        # plot the predicted vs experimental ddg values for the training, internal test, and external test sets
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], '--', lw=2, c='gray')

        ax.set_xlabel('Experimental $ΔΔG^{‡}$', fontsize=16)
        ax.set_ylabel('Predicted $ΔΔG^{‡}$ (kcal/mol)', fontsize=16)
        ax.scatter(y_train, y_train_pred, color='black', label='Train', alpha=0.7)
        ax.scatter(y_internal_test, y_internal_test_pred, color='#722378', label='Internal Test', alpha=0.7)
        ax.scatter(y_external_test, y_external_test_pred, color='#BD5077', label='External Test', alpha=0.7)
        ax.set_xlim(-0.7, 1.9)
        ax.set_ylim(-0.7, 1.9)
        ax.legend()

        plt.show()

    return selected_feature_names, (r2_train, r2_internal_test, r2_external_test), (mae_train, mae_internal_test, mae_external_test)

#### Recursive Feature Elimination with Cross-Validation (RFECV) ####
def rfecv_feature_selection(df, threshold=None, scaler=None, 
                            rfecv_reg=Ridge(alpha=0.3), rfecv_step=1, rfecv_cv=KFold(5, shuffle=True, random_state=42), rfecv_scoring='neg_mean_squared_error', rfecv_min_features_to_select=2,
                            fit_regressor=KernelRidge(kernel='rbf', alpha=0.7),  
                            plot=True,
                            add_rxn_ohe=False):
    """
    Perform feature selection using Recursive Feature Elimination with Cross-Validation (RFECV) and evaluate the performance of a specified regressor on the selected features.
    Parameters:
        # preprocess the data
    df (pd.DataFrame): The input dataframe containing the features and target variable.
    threshold (float): The threshold for removing collinear features based on R^2, if threshold is None, no features will be removed.
    scaler (sklearn.preprocessing.StandardScaler): The scaler to standardize the features.
        # RFECV parameters
    rfecv_reg (sklearn regressor): The regressor to use for RFECV (default: Ridge(alpha=0.3)).
    rfecv_step (int): The number of features to remove at each iteration of RFECV (default: 1).
    rfecv_cv (sklearn.model_selection.KFold): The cross-validation strategy for RFECV (default: KFold(5, shuffle=True, random_state=42)).
    rfecv_scoring (str): The scoring metric for RFECV (default: 'neg_mean_squared_error').
    rfecv_min_features_to_select (int): The minimum number of features to select with RFECV (default: 2).
        # Regressor fitting and evaluation
    fit_regressor (sklearn regressor): The regressor to fit on the selected features and evaluate performance (default: KernelRidge(kernel='rbf', alpha=0.7)).
        # plot
    plot (bool): Whether to plot the predicted vs experimental values for the training, internal test, and external test sets (default: True).
        # add_rxn_ohe (bool): Whether to add the one-hot encoded 'rxn' feature to the selected features (default: False).
    Returns:
    tuple: A tuple containing the selected feature names, R2 scores, and MAE scores for the training, internal test, and external test sets."""

    # preprocess the data
    X_train, y_train, _, X_internal_test, y_internal_test, _ , X_external_test, y_external_test, _ = preprocess_data(df, threshold=threshold, scaler=scaler, silent=True)

    rfecv = RFECV(
        estimator=rfecv_reg,
        step=rfecv_step,
        cv=rfecv_cv,
        scoring=rfecv_scoring,        
        min_features_to_select=rfecv_min_features_to_select)
    
    _ = rfecv.fit_transform(X_train, y_train)
    
    selected_feature_indices = rfecv.get_support(indices=True)
    selected_feature_names = X_train.columns[selected_feature_indices]

    if add_rxn_ohe and "rxn" not in selected_feature_names:
        selected_feature_names = selected_feature_names.tolist() + ["rxn"]

    X_train_selected = X_train[selected_feature_names]
    X_internal_test_selected = X_internal_test[selected_feature_names]
    X_external_test_selected= X_external_test[selected_feature_names]

    if plot:
        print("Optimal number of features:", rfecv.n_features_)
        print("Selected feature names:", selected_feature_names)

    fit_regressor.fit(X_train_selected, y_train)
    y_train_pred = fit_regressor.predict(X_train_selected)
    y_internal_test_pred = fit_regressor.predict(X_internal_test_selected)
    y_external_test_pred = fit_regressor.predict(X_external_test_selected)

    r2_train = r2_score(y_train, y_train_pred)
    r2_internal_test = r2_score(y_internal_test, y_internal_test_pred)
    r2_external_test = r2_score(y_external_test, y_external_test_pred)

    mae_train = np.mean(np.abs(y_train - y_train_pred))
    mae_internal_test = np.mean(np.abs(y_internal_test - y_internal_test_pred))
    mae_external_test = np.mean(np.abs(y_external_test - y_external_test_pred))
    
    if plot:
        print (f"R2 Train: {round(r2_train, 2)}, R2 Validation: {round(r2_internal_test, 2)}, R2 Test: {round(r2_external_test, 2)}")
        print (f"MAE Train: {round(mae_train, 2)}, MAE Validation: {round(mae_internal_test, 2)}", f"MAE Test: {round(mae_external_test, 2)}")
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], '--', lw=2, c='gray')
        ax.set_xlabel('Experimental $ΔΔG^{‡}$', fontsize=16)
        ax.set_ylabel('Predicted $ΔΔG^{‡}$ (kcal/mol)', fontsize=16)
        ax.scatter(y_train, y_train_pred, color='black', label='Train', alpha=0.7)
        ax.scatter(y_internal_test, y_internal_test_pred, color='#722378', label='Internal Test', alpha=0.7)
        ax.scatter(y_external_test, y_external_test_pred, color='#BD5077', label='External Test', alpha=0.7)
        ax.set_xlim(-0.7, 1.9)
        ax.set_ylim(-0.7, 1.9)
        ax.legend()
        plt.show()

    return selected_feature_names, (r2_train, r2_internal_test, r2_external_test), (mae_train, mae_internal_test, mae_external_test)

#### Recursive Feature Elimination (RFE) ####
def rfe_feature_selection(df, threshold=None, scaler=None, 
                            rfecv_reg=Ridge(alpha=0.3), rfecv_step=1, rfecv_min_features_to_select=4,
                            fit_regressor=KernelRidge(kernel='polynomial', alpha=0.7), 
                            shuffle=False, random_state=42, 
                            plot=True,
                            add_rxn_ohe=False):
    """
    Perform feature selection using Recursive Feature Elimination with Cross-Validation (RFECV) and evaluate the performance of a specified regressor on the selected features.
    Parameters:
        # preprocess the data
    df (pd.DataFrame): The input dataframe containing the features and target variable.
    threshold (float): The threshold for removing collinear features based on R^2, if threshold is None, no features will be removed.
    scaler (sklearn.preprocessing.StandardScaler): The scaler to standardize the features.
        # RFECV parameters
    rfecv_reg (sklearn regressor): The regressor to use for RFECV (default: Ridge(alpha=0.3)).
    rfecv_step (int): The number of features to remove at each iteration of RFECV (default: 1).
    rfecv_min_features_to_select (int): The minimum number of features to select with RFECV (default: 2).
        # Regressor fitting and evaluation
    fit_regressor (sklearn regressor): The regressor to fit on the selected features and evaluate performance (default: KernelRidge(kernel='rbf', alpha=0.7)).
        # shuffle and random state
    shuffle (bool): Whether to shuffle the data before splitting (default: False).
    random_state (int): The random seed for reproducibility (default: 42).
        # plot
    plot (bool): Whether to plot the predicted vs experimental values for the training, internal test, and external test sets (default: True).
    add_rxn_ohe (bool): Whether to add reaction one-hot encoding features (default: False).
    Returns:
    tuple: A tuple containing the selected feature names, R2 scores, and MAE scores for the training, internal test, and external test sets."""

    # preprocess the data
    X_train, y_train, _, X_internal_test, y_internal_test, _ , X_external_test, y_external_test, _ = preprocess_data(df, threshold=threshold, scaler=scaler, silent=True)
    
    # shuffle the data if specified
    if shuffle:
        rng = np.random.RandomState(42)
        shuffled_cols = rng.permutation(X_train.columns)
        X_train = X_train[shuffled_cols]
        X_internal_test = X_internal_test[shuffled_cols]
        X_external_test = X_external_test[shuffled_cols]

    rfecv = RFE(
        estimator=rfecv_reg,
        step=rfecv_step,
        n_features_to_select=rfecv_min_features_to_select)

    
    _ = rfecv.fit_transform(X_train, y_train)
    selected_feature_indices = rfecv.get_support(indices=True)
    selected_feature_names = X_train.columns[selected_feature_indices]

    if add_rxn_ohe and "rxn" not in selected_feature_names:
        selected_feature_names = selected_feature_names.tolist() + ["rxn"]
    X_train_selected = X_train[selected_feature_names]
    X_internal_test_selected = X_internal_test[selected_feature_names]
    X_external_test_selected= X_external_test[selected_feature_names]


    if plot:
        print("Optimal number of features:", rfecv.n_features_)
        print("Selected feature names:", selected_feature_names)
        print(f"Selected features: {selected_feature_names}")


    fit_regressor.fit(X_train_selected, y_train)
    y_train_pred = fit_regressor.predict(X_train_selected)
    y_internal_test_pred = fit_regressor.predict(X_internal_test_selected)
    y_external_test_pred = fit_regressor.predict(X_external_test_selected)

    r2_train = r2_score(y_train, y_train_pred)
    r2_internal_test = r2_score(y_internal_test, y_internal_test_pred)
    r2_external_test = r2_score(y_external_test, y_external_test_pred)

    mae_train = np.mean(np.abs(y_train - y_train_pred))
    mae_internal_test = np.mean(np.abs(y_internal_test - y_internal_test_pred))
    mae_external_test = np.mean(np.abs(y_external_test - y_external_test_pred))
    
    if plot:
        print (f"R2 Train: {round(r2_train, 2)}, R2 Validation: {round(r2_internal_test, 2)}, R2 Test: {round(r2_external_test, 2)}")
        print (f"MAE Train: {round(mae_train, 2)}, MAE Validation: {round(mae_internal_test, 2)}", f"MAE Test: {round(mae_external_test, 2)}")
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], '--', lw=2, c='gray')
        ax.set_xlabel('Experimental $ΔΔG^{‡}$', fontsize=16)
        ax.set_ylabel('Predicted $ΔΔG^{‡}$ (kcal/mol)', fontsize=16)
        ax.scatter(y_train, y_train_pred, color='black', label='Train', alpha=0.7)
        ax.scatter(y_internal_test, y_internal_test_pred, color='#722378', label='Internal Test', alpha=0.7)
        ax.scatter(y_external_test, y_external_test_pred, color='#BD5077', label='External Test', alpha=0.7)
        ax.set_xlim(-0.7, 1.9)
        ax.set_ylim(-0.7, 1.9)
        ax.legend()
        plt.show()

    return selected_feature_names, (r2_train, r2_internal_test, r2_external_test), (mae_train, mae_internal_test, mae_external_test)



In [ ]:
######## ------  defining preprocessing and feature selection functions ------ ########
def list_correlated_features_to_remove(df, threshold):
    """List a set of correlated features to remove based on a threshold, keeps one element of each correlated pair."""
    df_num = df.select_dtypes(include=[np.number])  # Select only numeric columns
    corr = df_num.corr().abs()
    to_drop = set()
    for i, col1 in enumerate(df_num.columns):
        for col2 in df_num.columns[i+1:]:
            if corr.loc[col1, col2] > threshold:
                to_drop.add(col2)  # Keep col1, drop col2

    to_keep = [col for col in df_num.columns if col not in to_drop]
    return to_drop, to_keep

def get_Xy_train_val_test(df_exp, df_features_scaled):
    df_scaled = pd.concat([df_exp.reset_index(drop=True), df_features_scaled.reset_index(drop=True)], axis=1)
    X = df_features_scaled
    y = (df_scaled['ddg_flipped'])
    y = pd.to_numeric(y, errors='coerce').astype(float)
    y_labels = df_scaled['ligandID']
    return X, y, y_labels

# Train|Validation|Test is done on the basis of the set column in the spreadsheet (y-equidistant sampling).
def preprocess_data(df, threshold=None, scaler=None, silent=False):
    """Preprocess the data by removing collinear features and standardizing the features.
    Args:
        df (pd.DataFrame): The input dataframe containing the features and target variable.
        threshold (float): The threshold for removing collinear features based on R^2, if threshold is None, no features will be removed.
        scaler (sklearn.preprocessing.StandardScaler): The scaler to standardize the features.
    Returns:
        X_train (pd.DataFrame): The preprocessed training features.
        y_train (pd.Series): The target variable for the training set.
        y_train_labels (pd.Series): The labels for the training set.
        X_internal_test (pd.DataFrame): The preprocessed internal test features.
        y_internal_test (pd.Series): The target variable for the internal test set.
        y_internal_test_labels (pd.Series): The labels for the internal test set.
        X_external_test (pd.DataFrame): The preprocessed external test features.
        y_external_test (pd.Series): The target variable for the external test set.
        y_external_test_labels (pd.Series): The labels for the external test set."""
    
    # 0. Train|Validation|Test is done on the basis of the set column in the spreadsheet (y-equidistant sampling).
    df_train = df[df['set'] == 'train']
    df_internal_test = df[df['set'] == 'validation']
    df_external_test = df[df['set'] == 'test']

    # train
    df_exp_train = df_train.loc[:,:'ligand_class']
    df_features_train = df_train.loc[:,'HOMO_Boltz':]

    # validation
    df_exp_internal_test = df_internal_test.loc[:,:'ligand_class']
    df_features_internal_test = df_internal_test.loc[:,'HOMO_Boltz':]

    # test
    df_exp_external_test = df_external_test.loc[:,:'ligand_class']
    df_features_external_test = df_external_test.loc[:,'HOMO_Boltz':]

    # 1. remove collinear features r2 > threshold
    if threshold is not None:
        train_cor_cols, train_uncor_cols = list_correlated_features_to_remove(df_features_train, threshold)
        if not silent:
            print(f'Shape of descriptors file before removing parameters with r^2 > {threshold} :',df_features_train.shape)
        df_features_train = df_features_train[train_uncor_cols]
        df_features_internal_test = df_features_internal_test[train_uncor_cols]
        df_features_external_test = df_features_external_test[train_uncor_cols]
        if not silent:
            print(f'Shape of descriptors file after removing parameters with R^2 > {threshold} : ',df_features_train.shape)

    feature_columns_collinear_removed = df_features_train.columns

    # 2. standardize the features (using standard scaler)
    if scaler is not None:
        df_features_train_scaled            = pd.DataFrame(scaler.fit_transform(df_features_train), columns=feature_columns_collinear_removed)
        df_features_internal_test_scaled    = pd.DataFrame(scaler.transform(df_features_internal_test), columns=feature_columns_collinear_removed)
        df_features_external_test_scaled    = pd.DataFrame(scaler.transform(df_features_external_test), columns=feature_columns_collinear_removed)

    X_train, y_train, y_train_labels = get_Xy_train_val_test(df_exp_train, df_features_train_scaled)
    X_internal_test, y_internal_test, y_internal_test_labels = get_Xy_train_val_test(df_exp_internal_test, df_features_internal_test_scaled)
    X_external_test, y_external_test, y_external_test_labels = get_Xy_train_val_test(df_exp_external_test, df_features_external_test_scaled)

    return X_train, y_train, y_train_labels, X_internal_test, y_internal_test, y_internal_test_labels, X_external_test, y_external_test, y_external_test_labels

In [ ]:
######## ------  plot functions ------ ########
def parity_plot_for_best_model(model_info, ax, label, X_train, y_train, X_internal_test, y_internal_test, X_external_test, y_external_test):
    """Helper tool, to plot the parity plot for the best model based on the internal test R2 or MAE scores."""
    k_best, fitted_regressor, scoring_function, _, r2_scores, mae_scores = model_info

    selector = SelectKBest(score_func=scoring_function, k=k_best)
    X_train_sel = selector.fit_transform(X_train, y_train)
    X_internal_sel = selector.transform(X_internal_test)
    X_external_sel = selector.transform(X_external_test)

    model = fitted_regressor.__class__(**fitted_regressor.get_params())
    model.fit(X_train_sel, y_train)

    y_train_pred = model.predict(X_train_sel)
    y_internal_pred = model.predict(X_internal_sel)
    y_external_pred = model.predict(X_external_sel)

    y_all = np.concatenate([y_train.values, y_internal_test.values, y_external_test.values])
    y_min, y_max = y_all.min(), y_all.max()
    pad = 0.05 * (y_max - y_min)

    ax.plot([y_min - pad, y_max + pad], [y_min - pad, y_max + pad], '--', lw=2, c='gray')
    ax.scatter(y_train, y_train_pred, color='black', label='Train', alpha=0.7)
    ax.scatter(y_internal_test, y_internal_pred, color='#722378', label='Internal Test', alpha=0.7)
    ax.scatter(y_external_test, y_external_pred, color='#BD5077', label='External Test', alpha=0.7)

    ax.set_xlabel('Experimental $ΔΔG^{‡}$')
    ax.set_ylabel('Predicted $ΔΔG^{‡}$ (kcal/mol)')
    ax.set_xlim(y_min - pad, y_max + pad)
    ax.set_ylim(y_min - pad, y_max + pad)
    ax.set_title(
        f"{label}\n{k_best} features | {model.__class__.__name__} | {scoring_function.__name__}\n"
        f"Val R²={r2_scores[1]:.2f}, Val MAE={mae_scores[1]:.2f}"
    )
    ax.legend()

def parity_plot_for_rfecv_model(best_model_r2, df, silent=True):

    # Preprocess data with best threshold
    X_train, y_train, _, X_internal_test, y_internal_test, _, X_external_test, y_external_test, _ = preprocess_data(
        df, threshold=best_model_r2['threshold'], scaler=StandardScaler(), silent=silent
    )

    # Apply feature selection using RFECV with best parameters
    rfecv = RFECV(
        estimator=best_model_r2['rfecv_reg'],
        step=best_model_r2['rfecv_step'],
        cv=KFold(5, shuffle=True, random_state=42),
        scoring='neg_mean_squared_error',
        min_features_to_select=best_model_r2['min_features']
    )

    X_train_sel = rfecv.fit_transform(X_train, y_train)
    X_internal_test_sel = rfecv.transform(X_internal_test)
    X_external_test_sel = rfecv.transform(X_external_test)

    # Train model with best regressor
    best_regressor = best_model_r2['fit_regressor']
    best_regressor.fit(X_train_sel, y_train)

    # Generate predictions
    y_train_pred = best_regressor.predict(X_train_sel)
    y_internal_test_pred = best_regressor.predict(X_internal_test_sel)
    y_external_test_pred = best_regressor.predict(X_external_test_sel)

    print(f"Best RFECV model: {len(best_model_r2['features'])} features, R² (train, val, test): {[round(x,3) for x in best_model_r2['r2_scores']]}, MAE (train, val, test): {[round(x,3) for x in best_model_r2['mae_scores']]}")

    # Create parity plot
    fig, ax = plt.subplots(figsize=(8, 8))

    y_all = np.concatenate([y_train.values, y_internal_test.values, y_external_test.values])
    y_min, y_max = y_all.min(), y_all.max()
    pad = 0.1 * (y_max - y_min)

    ax.plot([y_min - pad, y_max + pad], [y_min - pad, y_max + pad], '--', lw=2, c='gray', label='Perfect prediction')
    ax.scatter(y_train, y_train_pred, color='black', label='Train', alpha=0.7, s=80)
    ax.scatter(y_internal_test, y_internal_test_pred, color='#722378', label='Validation', alpha=0.7, s=80)
    ax.scatter(y_external_test, y_external_test_pred, color='#BD5077', label='Test', alpha=0.7, s=80)

    ax.set_xlabel('Experimental ΔΔG‡ (kcal/mol)', fontsize=14)
    ax.set_ylabel('Predicted ΔΔG‡ (kcal/mol)', fontsize=14)
    ax.set_xlim(y_min - pad, y_max + pad)
    ax.set_ylim(y_min - pad, y_max + pad)
    ax.set_aspect('equal')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)

    best_model_name = best_regressor.__class__.__name__
    plt.title(f'{best_model_name} Performance\nVal R²={best_model_r2["r2_scores"][1]:.3f}, Val MAE={best_model_r2["mae_scores"][1]:.3f}', fontsize=14)
    plt.tight_layout()
    plt.show()

    

### Data loading for CSII

In [ ]:
df = pd.read_excel('../fluoride_modeling_properties_w_experimental_data.xlsx',
                   sheet_name='CS2_combined_ddg', header=1)

df.head(5)

### Preprocessing of the features

In [ ]:
threshold = 0.7 # remove collinear features r2 > 0.7
scaler    = StandardScaler()

X_train, y_train, y_train_labels, X_internal_test, y_internal_test, y_internal_test_labels, X_external_test, y_external_test, y_external_test_labels = preprocess_data(df, threshold=threshold, scaler=scaler)

### Defining Grids for KBest and RFE(CV) hyperparameters optimization and feature selection 

In [ ]:
# definition of a grid for feature selection and model parameters to search over
from itertools import product

scaler          = StandardScaler()

# grid for selectk-best feature selection and model parameters to search over
rxn_ohe_option  = [True, False]
threshold       = [0.7, None]        
k_list          = [1, 2, 3, 4]
fit_regressors  =   [KernelRidge(kernel='rbf', alpha=a) for a in [0.7, 1.0, 2.0]] + \
                    [KernelRidge(kernel='polynomial', alpha=a) for a in [0.7, 1.0, 2.0]] + \
                    [Ridge(alpha=a) for a in [0.1, 0.5, 0.7, 1.0]]
scoring_functions = [f_regression, mutual_info_regression]

param_grid_kbest = list(product(threshold, k_list, fit_regressors, rxn_ohe_option, scoring_functions))

# grid for RFE-CV and RFE feature selection and model parameters to search over
rxn_ohe_option      = [True, False]
thresholds          = [None, 0.7]
rfecv_regs          = [Ridge(alpha=a) for a in [0.1, 0.3, 0.5, 1.0]]
rfecv_steps         = [1, 2]
rfecv_min_features  = [2, 3, 4, 5]
fit_regressors      =   [KernelRidge(kernel='rbf', alpha=a) for a in [0.7, 1.0, 2.0]] + \
                        [KernelRidge(kernel='polynomial', alpha=a) for a in [0.7, 1.0, 2.0]] + \
                        [Ridge(alpha=a) for a in [0.1, 0.5, 0.7, 1.0]]

param_grid = list(product(thresholds, rfecv_regs, rfecv_steps, rfecv_min_features, fit_regressors, rxn_ohe_option))

### Selection of best features using SelectKBest

In [ ]:

# Grid search over the defined hyperparameters to find the best model based on the internal test R2 and MAE scores.
best_model_r2_kbest = None
best_model_mae_kbest = None
best_r2_internal_test = -np.inf
best_mae_internal_test = np.inf

configs_selectkbest = []
for threshold_value, k, regressor, rxn_ohe_option, scoring_function in param_grid_kbest:
    X_train, y_train, y_train_labels, X_internal_test, y_internal_test, y_internal_test_labels, X_external_test, y_external_test, y_external_test_labels = preprocess_data(
        df.copy(), threshold=threshold_value, scaler=scaler, silent=True
    )
    selected_features, r2_scores, mae_scores = select_k_best_features(
        X_train, y_train, X_internal_test, y_internal_test, X_external_test, y_external_test,
        k, scoring_function, regressor, plot=False, add_rxn_ohe=rxn_ohe_option
    )
    r2_train, r2_internal_test, r2_external_test = r2_scores
    mae_train, mae_internal_test, mae_external_test = mae_scores
    
    if r2_internal_test > best_r2_internal_test:
        best_r2_internal_test = r2_internal_test
        best_model_r2_kbest = (k, regressor, scoring_function, selected_features, r2_scores, mae_scores)

    if mae_internal_test < best_mae_internal_test:
        best_mae_internal_test = mae_internal_test
        best_model_mae_kbest = (k, regressor, scoring_function, selected_features, r2_scores, mae_scores)

    configs_selectkbest.append({"k": k, "regressor": regressor, "scoring_function": scoring_function, "selected_features": selected_features, "r2_scores": r2_scores, "mae_scores": mae_scores})


# Print the best models based on internal test R2 and MAE scores, along with their selected features and performance metrics.
def fmt_scores(scores, ndigits=2):
    return '[' + ', '.join(f"{float(x):.{ndigits}f}" for x in scores) + ']'

def fmt_model_with_params(model):
    params = model.get_params()
    params_str = ', '.join(f"{k}={v}" for k, v in sorted(params.items()))
    return f"{model.__class__.__name__}({params_str})"

print("Best model based on validation R²:")
print(f"  k={best_model_r2_kbest[0]}")
print(f"  regressor={fmt_model_with_params(best_model_r2_kbest[1])}")
print(f"  scoring_function={best_model_r2_kbest[2].__name__}")
print(f"  {len(best_model_r2_kbest[3])} features selected: {list(best_model_r2_kbest[3])}")
print(f"  R² scores (train, val, test): {fmt_scores(best_model_r2_kbest[4])}")
print(f"  MAE scores (train, val, test): {fmt_scores(best_model_r2_kbest[5])}\n")

print("Best model based on validation MAE:")
print(f"  k={best_model_mae_kbest[0]}")
print(f"  regressor={fmt_model_with_params(best_model_mae_kbest[1])}")
print(f"  scoring_function={best_model_mae_kbest[2].__name__}")
print(f"  {len(best_model_mae_kbest[3])} features selected: {list(best_model_mae_kbest[3])}")
print(f"  R² scores (train, val, test): {fmt_scores(best_model_mae_kbest[4])}")
print(f"  MAE scores (train, val, test): {fmt_scores(best_model_mae_kbest[5])}")

# plot the parity plots for the best models based on internal test R2 and MAE scores.
fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)

parity_plot_for_best_model(best_model_r2_kbest, axes[0], 'Best by Validation R²', X_train, y_train, X_internal_test, y_internal_test, X_external_test, y_external_test)
parity_plot_for_best_model(best_model_mae_kbest, axes[1], 'Best by Validation MAE', X_train, y_train, X_internal_test, y_internal_test, X_external_test, y_external_test)

plt.tight_layout()
plt.show()

### Selection of best features using RFECV

In [ ]:
max_features = 4 # this parameter is used to discard any model that would use more than 4 features (rxn encoding excluded), to keep the models interpretable and avoid overfitting given the small dataset size.

# Grid search with parallelization
best_model_r2_rfecv = None
best_model_mae_rfecv = None
best_r2_val = -np.inf
best_mae_val = np.inf

total = len(param_grid)

def evaluate_rfecv_config(cfg):
    threshold, rfecv_reg, step, min_feat, fit_reg, rxn_ohe_option = cfg
    try:
        feats, r2s, maes = rfecv_feature_selection(
            df,
            threshold=threshold,
            scaler=StandardScaler(),
            rfecv_reg=clone(rfecv_reg),
            rfecv_step=step,
            rfecv_cv=KFold(5, shuffle=True, random_state=42),
            rfecv_scoring='neg_mean_squared_error',
            rfecv_min_features_to_select=min_feat,
            fit_regressor=clone(fit_reg),
            plot=False,
            add_rxn_ohe=rxn_ohe_option
        )
    except Exception:
        return None

    return {
        'threshold': threshold,
        'rfecv_reg': rfecv_reg,
        'rfecv_step': step,
        'min_features': min_feat,
        'fit_regressor': fit_reg,
        'features': feats,
        'r2_scores': r2s,
        'mae_scores': maes,
    }

results = Parallel(n_jobs=-1, prefer='processes')(
    delayed(evaluate_rfecv_config)(cfg) for cfg in tqdm(param_grid, total=total, desc='RFECV grid search (parallel dispatch)')
)

configs_rfecv = [r for r in results if r is not None]

for config in configs_rfecv:
    if config is None:
        continue
    
    # Discard models with more than max_features (excluding rxn encoding)
    if rxn_ohe_option and len(config['features']) > max_features + 1:
        continue
    elif not rxn_ohe_option and len(config['features']) > max_features:
        continue

    # Store best model information based on validation R2 and MAE scores
    _, r2_val, _ = config['r2_scores']
    _, mae_val, _ = config['mae_scores']

    if r2_val > best_r2_val:
        best_r2_val = r2_val
        best_model_r2_rfecv = config

    if mae_val < best_mae_val:
        best_mae_val = mae_val
        best_model_mae_rfecv = config


# Print summary of results
evaluated = sum(r is not None for r in results)
print(f"\nEvaluated {evaluated}/{total} configurations\n")

for label, best in [("Best by Validation R²", best_model_r2_rfecv), ("Best by Validation MAE", best_model_mae_rfecv)]:
    if best is None:
        continue
    print(f"--- {label} ---")
    print(f"  threshold={best['threshold']}, rfecv_reg={best['rfecv_reg']}, step={best['rfecv_step']}, min_features={best['min_features']}")
    print(f"  fit_regressor={best['fit_regressor']}")
    print(f"  {len(best['features'])} features: {best['features']}")
    print(f"  R² (train, val, test): {[round(x,3) for x in best['r2_scores']]}")
    print(f"  MAE (train, val, test): {[round(x,3) for x in best['mae_scores']]}\n")

# Parity plots for best RFECV models
parity_plot_for_rfecv_model(best_model_r2_rfecv, df)
parity_plot_for_rfecv_model(best_model_mae_rfecv, df)

### Selection of best features using RFE

In [ ]:
X_train, y_train, y_train_labels, X_internal_test, y_internal_test, y_internal_test_labels, X_external_test, y_external_test, y_external_test_labels = preprocess_data(df, threshold=None, scaler=StandardScaler(), silent=True)

# Grid search over RFE feature selection parameters
max_features = 4 # without rxn encoding

best_model_r2_rfe = None
best_model_mae_rfe = None
best_r2_val = -np.inf
best_mae_val = np.inf
total = len(param_grid)

def evaluate_rfe_config(cfg):
    threshold, rfecv_reg, step, min_feat, fit_reg, rxn_ohe = cfg
    try:
        feats, r2s, maes = rfe_feature_selection(
            df,
            threshold=threshold,
            scaler=StandardScaler(),
            rfecv_reg=clone(rfecv_reg),
            rfecv_step=step,
            rfecv_min_features_to_select=min_feat,
            fit_regressor=clone(fit_reg),
            plot=False,
            add_rxn_ohe=rxn_ohe,
        )
    except Exception:
        return None

    return {
        'threshold': threshold,
        'rfecv_reg': rfecv_reg,
        'rfecv_step': step,
        'min_features': min_feat,
        'fit_regressor': fit_reg,
        'rxn_ohe': rxn_ohe,
        'features': feats,
        'r2_scores': r2s,
        'mae_scores': maes,
    }

results = Parallel(n_jobs=-1, prefer='processes')(
    delayed(evaluate_rfe_config)(cfg) for cfg in tqdm(param_grid, total=total, desc='RFE grid search (parallel dispatch)')
)

configs_rfe = [r for r in results if r is not None]

for config in configs_rfe:
    _, r2_val, _ = config['r2_scores']
    _, mae_val, _ = config['mae_scores']

    if config['rxn_ohe'] and len(config['features']) > max_features + 1:
        continue
    if not config['rxn_ohe'] and len(config['features']) > max_features:
        continue

    if r2_val > best_r2_val:
        best_r2_val = r2_val
        best_model_r2_rfe = config

    if mae_val < best_mae_val:
        best_mae_val = mae_val
        best_model_mae_rfe = config

evaluated = len(configs_rfe)
print(f"\nEvaluated {evaluated}/{total} configurations\n")

for label, best in [("Best by Validation R²", best_model_r2_rfe), ("Best by Validation MAE", best_model_mae_rfe)]:
    if best is None:
        continue
    print(f"--- {label} ---")
    print(f"  threshold={best['threshold']}, rfecv_reg={best['rfecv_reg']}, step={best['rfecv_step']}, min_features={best['min_features']}, add_rxn_ohe={best['rxn_ohe']}")
    print(f"  fit_regressor={best['fit_regressor']}")
    print(f"  {len(best['features'])} features: {best['features']}")
    print(f"  R² (train, val, test): {[round(float(x), 3) for x in best['r2_scores']]}")
    print(f"  MAE (train, val, test): {[round(float(x), 3) for x in best['mae_scores']]}\n")


parity_plot_for_rfecv_model(best_model_r2_rfe, df)
parity_plot_for_rfecv_model(best_model_mae_rfe, df)